# FwP: EDA, PR Curve & Feature Importance
Notebook reads either production retraining features or synthetic fallback.

In [ ]:
import os, sys, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay
sys.path.insert(0, '..')
from src.models.train import FEATURE_COLUMNS, prepare_data, train_lightgbm, load_config
config = load_config('../configs/production.yaml') if os.path.exists('../data/production/training_features.csv') else load_config('../configs/lightgbm.yaml')
df = pd.read_csv(config['data_path'])
df.describe().T

In [ ]:
X_train, y_train, X_val, y_val, _, _ = prepare_data(config)
model = train_lightgbm(X_train, y_train, X_val, y_val, config)
PrecisionRecallDisplay.from_predictions(y_val, model.predict_proba(X_val)[:, 1]); plt.title('Validation PR curve'); plt.show()
pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values().plot.barh(title='LightGBM feature importance'); plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(model)
shap.summary_plot(explainer.shap_values(X_val), X_val, feature_names=FEATURE_COLUMNS)